# gpt-oss-20b — single-turn chat with a system prompt

Minimal notebook: set a system prompt, send one user turn, print the model's reply.

Runs on the same RunPod pod as `role-probe-demo_runpod.ipynb` (RTX PRO 4500 Blackwell, 32 GB).
See memory `runpod-setup` for pod/volume details. Before running:

1. Start the pod if it's stopped.
2. `bash /workspace/prompt-injection-as-role-confusion/setup_python_runpod.sh --fast` over SSH (restores the venv in ~1-2 min).
3. In JupyterLab, select the **"Role analysis (uv)"** kernel (not the stock "Python 3" kernel — it's missing `kernels`/`triton` and MXFP4 will silently dequantize to bf16, which needs ~48 GB).

In [1]:
"""
Hardware + MXFP4 preflight. Run before spending time downloading a 13 GB model.
"""
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # must be set before torch allocates

import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU visible.')

p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name} | {p.total_memory / 1024 ** 3:.1f} GB | compute capability {p.major}.{p.minor}')
if (p.major, p.minor) < (7, 5):
    raise RuntimeError('Compute capability < 7.5: MXFP4 unavailable, model will not fit.')

import importlib
for _mod in ['kernels', 'triton']:
    _m = importlib.import_module(_mod)
    print(f'import {_mod}: OK ({getattr(_m, "__version__", "?")})')

from transformers.utils import is_kernels_available
if not is_kernels_available():
    from transformers.utils import import_utils as _iu
    raise RuntimeError(
        'transformers cannot use the kernels package, so it would dequantize gpt-oss-20b to '
        f'bf16 (~48 GB). Install kernels within [{_iu.KERNELS_MIN_VERSION}, {_iu.KERNELS_MAX_VERSION}) '
        'and restart the kernel (is_kernels_available() is cached).'
    )
print('MXFP4 prerequisites OK.')

GPU: NVIDIA RTX PRO 4500 Blackwell | 31.4 GB | compute capability 12.0


import kernels: OK (0.15.2)
import triton: OK (3.5.1)


MXFP4 prerequisites OK.


## 1. Load model + tokenizer

In [2]:
# dtype MUST stay 'auto' — naming a concrete dtype dequantizes MXFP4 to 16-bit (~48 GB).
# device_map pins everything to cuda:0 so an out-of-memory model raises a clean OOM
# instead of silently offloading to CPU.
CACHE_DIR = '/workspace/hf'  # or None if uncached

from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    'openai/gpt-oss-20b',
    cache_dir = CACHE_DIR,
    attn_implementation = 'eager',
    dtype = 'auto',
    device_map = 'cuda:0',
).eval()
tokenizer = AutoTokenizer.from_pretrained('openai/gpt-oss-20b', cache_dir = CACHE_DIR)

# Sanity-check the weights actually stayed 4-bit (get_memory_footprint() under-reports MXFP4).
allocated = torch.cuda.memory_allocated(0) / 1024 ** 3
print(f'Allocated on GPU: {allocated:.1f} GB')
if allocated > 20:
    raise RuntimeError('Well above the ~13 GB MXFP4 size - the weights were dequantized and this will OOM.')

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 43 files:   0%|          | 0/43 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

Allocated on GPU: 12.8 GB


## 2. Set the system prompt + user message

This builds the raw Harmony-formatted prompt by hand instead of `tokenizer.apply_chat_template`.
`apply_chat_template` never lets a caller-supplied message land in the actual `<|start|>system<|message|>`
channel — it always fills that channel itself from template variables (see `chat_template.jinja`'s
`build_system_message()` macro) and routes anything passed as `role: "system"` into a *separate*
`developer` channel instead.

**Note on an earlier version of this cell**: a first pass tried isolating which piece of the
auto-generated boilerplate (identity/cutoff/date/reasoning/valid-channels) was needed to stop the
model looping in the `analysis` channel. That comparison turned out to be unreliable — gpt-oss's
default MoE routing is non-deterministic across process runs (see `role-probe-demo_runpod.ipynb`'s
`DETERMINISTIC_EXPERTS` note), so re-loading the model fresh for each variant meant identical prompt
text could legitimately produce different generations for reasons having nothing to do with the
prompt content. Forcing deterministic (eager) expert routing to fix that is risky here — it can
dequantize the MoE expert weights to 16-bit, which is what makes MXFP4 fit in 32 GB in the first
place. So instead: both prompt variants are generated below in the *same* process, from the *same*
loaded model, each repeated twice — a fair within-run comparison without touching quantization.

In [3]:
from datetime import date

MODEL_IDENTITY = "You are ChatGPT, a large language model trained by OpenAI."
KNOWLEDGE_CUTOFF = "2024-06"
CURRENT_DATE = date.today().isoformat()
REASONING_EFFORT = "medium"
VALID_CHANNELS_LINE = "# Valid channels: analysis, commentary, final. Channel must be included for every message."
CUSTOM_INSTRUCTION = "You are a concise, helpful assistant."

BOILERPLATE = (
    f"{MODEL_IDENTITY}\n"
    f"Knowledge cutoff: {KNOWLEDGE_CUTOFF}\n"
    f"Current date: {CURRENT_DATE}\n\n"
    f"Reasoning: {REASONING_EFFORT}\n\n"
    f"{VALID_CHANNELS_LINE}"
)

# The system prompt actually used going forward: full boilerplate (this is what the model was
# trained to see in this channel) plus your own instruction appended - edit CUSTOM_INSTRUCTION freely.
SYSTEM_PROMPT = f"{BOILERPLATE}\n\n{CUSTOM_INSTRUCTION}"

# Kept only for the reproducibility check below: your instruction with NO boilerplate at all.
BARE_SYSTEM_PROMPT = CUSTOM_INSTRUCTION

USER_MESSAGE = "What's an interesting cat fact?"

def build_raw_prompt(system_content: str, user_message: str) -> str:
    return (
        f"<|start|>system<|message|>{system_content}<|end|>"
        f"<|start|>user<|message|>{user_message}<|end|>"
        "<|start|>assistant"
    )

raw_prompt = build_raw_prompt(SYSTEM_PROMPT, USER_MESSAGE)
print(raw_prompt)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-08-07

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.

You are a concise, helpful assistant.<|end|><|start|>user<|message|>What's an interesting cat fact?<|end|><|start|>assistant


## 3. Run a single turn

In [4]:
import re

# add_special_tokens=False: the <|start|>/<|message|>/<|end|> markers above are already the literal
# text the tokenizer maps to Harmony's special token ids, same as render_single_role_gptoss's approach.
def extract_channel(text: str, channel: str) -> str | None:
    m = re.search(rf"<\|channel\|>{channel}<\|message\|>(.*?)(?:<\|end\|>|<\|return\|>|$)", text, re.DOTALL)
    return m.group(1).strip() if m else None

def run_turn(system_content: str, user_message: str, max_new_tokens: int = 512) -> dict:
    prompt = build_raw_prompt(system_content, user_message)
    inputs = tokenizer(prompt, return_tensors = 'pt', add_special_tokens = False).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens = max_new_tokens, do_sample = False)
    new_tokens = output_ids[0, inputs['input_ids'].shape[1]:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens = False)
    final = extract_channel(raw_output, 'final')
    return {
        'raw_output': raw_output,
        'analysis': extract_channel(raw_output, 'analysis'),
        'final': final,
        'reached_final': final is not None,
        'hit_token_cap': len(new_tokens) >= max_new_tokens,
    }

# Reproducibility check: same prompt, same loaded model, repeated twice - do we get the same
# reached_final / hit_token_cap outcome both times? Then compare the boilerplate vs bare variant.
for label, system_content in [('FULL (boilerplate + instruction)', SYSTEM_PROMPT), ('BARE (instruction only)', BARE_SYSTEM_PROMPT)]:
    for rep in (1, 2):
        result = run_turn(system_content, USER_MESSAGE)
        print(f'[{label} | rep {rep}] reached_final={result["reached_final"]} hit_token_cap={result["hit_token_cap"]}')
        print('  final:', (result['final'] or '(none)')[:200])
    print()

# The actual single turn this notebook reports: full boilerplate + your instruction.
result = run_turn(SYSTEM_PROMPT, USER_MESSAGE)
print('[RAW OUTPUT]')
print(result['raw_output'])
print()
print('[ANALYSIS / CoT]')
print(result['analysis'] or '(none)')
print()
print('[FINAL RESPONSE]')
print(result['final'] or '(model never reached the final channel - see raw output above)')

[FULL (boilerplate + instruction) | rep 1] reached_final=True hit_token_cap=False
  final: Did you know that cats can make over **100 different vocal sounds**? That’s more than dogs, which typically use only about 10. From chirps and trills to the classic “meow,” cats have a surprisingly ve


[FULL (boilerplate + instruction) | rep 2] reached_final=True hit_token_cap=False
  final: Did you know that cats can make over **100 different vocal sounds**? That’s more than dogs, which typically use only about 10. From chirps and trills to the classic “meow,” cats have a surprisingly ve



[BARE (instruction only) | rep 1] reached_final=False hit_token_cap=True
  final: (none)


[BARE (instruction only) | rep 2] reached_final=False hit_token_cap=True
  final: (none)



[RAW OUTPUT]
<|channel|>analysis<|message|>We need to give an interesting cat fact. Keep it concise. Provide a fact. Maybe about cat whiskers, or their purring, or their sense of hearing, or their ability to land on feet. Let's pick something interesting: cats have a unique "righting reflex" that allows them to land on their feet. Or that cats have a special organ in their ears that helps them detect high-frequency sounds. Or that cats have a "third eyelid" called the nictitating membrane. Or that cats can make over 100 different vocal sounds, while dogs only about 10. That is interesting. Let's give that.<|end|><|start|>assistant<|channel|>final<|message|>Did you know that cats can make over **100 different vocal sounds**? That’s more than dogs, which typically use only about 10. From chirps and trills to the classic “meow,” cats have a surprisingly versatile vocal toolbox to communicate with humans and other felines.<|return|>

[ANALYSIS / CoT]
We need to give an interesting cat fact